# aule — Classification metrics

This notebook covers `aule.metrics.classification`: metrics for **segmentation masks and categorical label maps** (e.g. cloud masks, land cover classification, change maps). These functions are designed to be generic: they work both for binary masks (`average='binary'`, values in `{0, 1}`) and for multi-class integer label maps (`average='macro'` or `'micro'`, with an optional `num_classes`).

In [ ]:
!pip install aule

## Setup: binary masks

Let's start with a simple binary segmentation case (e.g. cloud / no-cloud).

In [ ]:
import numpy as np

np.random.seed(3)

gt_mask   = (np.random.rand(64, 64, 1) > 0.6).astype(int)
# simulate an imperfect prediction: agrees most of the time, but with some errors
flip_mask = np.random.rand(*gt_mask.shape) < 0.1
pred_mask = np.where(flip_mask, 1 - gt_mask, gt_mask)

print('Fraction of foreground in GT:  ', gt_mask.mean())
print('Fraction of foreground in pred:', pred_mask.mean())

## IoU and Dice

The two most common overlap metrics for binary segmentation. Both are 1.0 for a perfect match, and both return 1.0 (by convention) when both masks are empty.

In [ ]:
from aule.metrics import iou, dice

print('IoU: ', iou(gt_mask, pred_mask))
print('Dice:', dice(gt_mask, pred_mask))

# sanity check: perfect match
print('IoU (identical):', iou(gt_mask, gt_mask))

## Precision, recall, F1

Useful when false positives and false negatives have different costs (e.g. missing a cloud vs. false-flagging clear sky as cloud).

In [ ]:
from aule.metrics import precision_recall_f1

scores = precision_recall_f1(gt_mask, pred_mask)
print('Precision:', scores['precision'])
print('Recall:   ', scores['recall'])
print('F1:       ', scores['f1'])

## Confusion-matrix-derived metrics

`confusion_matrix_metrics` gives the standard binary confusion matrix breakdown (TP/FP/TN/FN) plus accuracy, specificity, and balanced accuracy (averages sensitivity and specificity, useful for imbalanced classes).

In [ ]:
from aule.metrics import confusion_matrix_metrics

result = confusion_matrix_metrics(gt_mask, pred_mask)
for key, value in result.items():
    print(f'{key}: {value}')

## Multi-class: land cover example

Now let's simulate a 4-class land cover map (e.g. water, forest, urban, bare soil) and show how the same functions generalize via `num_classes` and `average`.

In [ ]:
class_names = ['water', 'forest', 'urban', 'bare soil']
num_classes = len(class_names)

gt_labels = np.random.randint(0, num_classes, (64, 64, 1))
# simulate predictions with ~70% agreement
agree_mask = np.random.rand(*gt_labels.shape) < 0.7
random_labels = np.random.randint(0, num_classes, gt_labels.shape)
pred_labels = np.where(agree_mask, gt_labels, random_labels)

print('Overall pixel agreement:', (gt_labels == pred_labels).mean())

## Multi-class IoU and Dice

`average='macro'` computes per-class IoU/Dice then averages unweighted (treats rare classes equally); `average='micro'` pools all classes' intersection/union before dividing (weights by class frequency).

In [ ]:
print('IoU (macro):', iou(gt_labels, pred_labels, num_classes=num_classes, average='macro'))
print('IoU (micro):', iou(gt_labels, pred_labels, num_classes=num_classes, average='micro'))
print('Dice (macro):', dice(gt_labels, pred_labels, num_classes=num_classes, average='macro'))

## Multi-class precision/recall/F1

In [ ]:
scores_macro = precision_recall_f1(gt_labels, pred_labels, num_classes=num_classes, average='macro')
print('Macro-averaged F1:', scores_macro['f1'])

## Cohen's Kappa

Kappa corrects for chance agreement, which matters a lot for multi-class problems with imbalanced classes — a high raw accuracy can still mean low kappa if one class dominates.

In [ ]:
from aule.metrics import cohen_kappa

score = cohen_kappa(gt_labels, pred_labels, num_classes=num_classes)
print('Cohen Kappa:', score)

# sanity check: completely random labels should give kappa close to 0
random_pred = np.random.randint(0, num_classes, gt_labels.shape)
print('Cohen Kappa (random prediction):', cohen_kappa(gt_labels, random_pred, num_classes=num_classes))

## Visualizing the confusion matrix

The dedicated `aule.plots.plot_confusion_matrix` function (shown in the plotting notebook) gives an annotated heatmap directly from the label maps.

In [ ]:
from aule.plots import plot_confusion_matrix

fig, ax = plot_confusion_matrix(gt_labels, pred_labels, num_classes=num_classes, class_names=class_names)